# Chapter 2: LightGBM — Built for Speed and Scale

In [ ]:
from lightgbm import LGBMClassifier

lgb_model = LGBMClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    num_leaves=31,           # LightGBM-SPECIFIC: controls tree complexity directly, not just via depth
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)
lgb_model.fit(X_train, y_train)

pred_lgb = lgb_model.predict_proba(X_test)[:, 1]
print("LightGBM ROC-AUC:", round(roc_auc_score(y_test, pred_lgb), 4))

The core structural difference from XGBoost, worth understanding conceptually even if you never need to tune around it directly: XGBoost grows trees **level-wise** (expanding every branch at the current depth before going deeper, keeping trees balanced); LightGBM grows trees **leaf-wise** (always splitting whichever single leaf currently reduces error the most, regardless of overall balance). Leaf-wise growth typically reaches a given accuracy with fewer splits — hence LightGBM's speed advantage — but can overfit faster on smaller datasets if left unchecked, which is exactly why `num_leaves` (directly capping tree complexity) is LightGBM's signature extra hyperparameter, used alongside `max_depth` rather than instead of it.

💡 **Practical guidance for choosing between XGBoost and LightGBM day to day:** on small-to-medium datasets (under a few hundred thousand rows — the overwhelming majority of business analytics work), performance differences between the two are usually marginal, and the choice comes down to team convention or personal preference. LightGBM's speed advantage becomes genuinely decisive once you're working with millions of rows or need to iterate through many tuning experiments quickly.

### Cheat Sheet — LightGBM

| Task | Code |
|---|---|
| Classifier | `LGBMClassifier(n_estimators, num_leaves, learning_rate, ...)` |
| Key extra hyperparameter | `num_leaves` (directly caps tree complexity — the LightGBM-specific knob) |
| Silence verbose logging | `verbose=-1` |

---